# Datamine CELLCONF Process Example

This notebook demonstrates how to configure and run the **`cellconf`** process wrapper in `dmstudio`.

## Process Description

## CELLCONF

**Note** : This is a _superprocess_ and running it may have an effect on other Datamine files in the project.

This process creates a summary table and graph of confidence for parent cells in a block model which has been created using conditional simulation.

The [CSMODEL](<csmodel.md>) process calculates very detailed information for confidence values of every parent cell in the simulated model. The CELLCONF process then summarizes this information into a table showing average confidence values for cells in different grade ranges. The following observations could potentially be derived from the results of this process:

For a cell whose estimated value lies between 1.5 and 1.75 g/t Au:

* there is a 10% probability that the actual grade will be less than 1.0 g/t;
* there is a 10% probability that it will exceed 2.23 g/t.

For a cell whose estimated value lies between 3.0 and 3.25 g/t:

* there is a 10% probability that the actual grade will be less than 2.06 g/t;
* there is a 10% probability that it will exceed 4.12 g/t.

### Input Model File

The input STATMOD model, previously created by the CSMODEL process, must include at least one percentile field, PCxx, where xx is the numeric percentile value. It must also include the field **MEAN** , the average of all the simulated values for each cell, also known as the **EType** estimated grade of the cell.

### Cutoffs

The required input includes a series of cutoff grades which define grade ranges for which average confidence values are calculated.

* If a regular set of cutoff grades is appropriate, then cutoffs can be defined using the CUTINT (cutoff interval) and CUTMAX (maximum cutoff) parameters. For example if CUTINT=0.5 and CUTMAX=3, then 7 grade ranges (bins) are defined as follows:

Minimum value |  Maximum value
---|---
0.00 |  0.50
0.50 |  1.00
1.00 |  1.50
1.50 |  2.00
2.00 |  2.50
2.50 |  3.00
4.00 |  ∞

* If irregular intervals are required, then cutoff values can be input from the CUTOFF file - the field name COGRADE must be used for cutoff grades.

* **Otherwise**:

* A zero cutoff value should not be included in the file.
* An extra bin from the maximum cutoff to 9999999 will always be automatically included for both the parameter and file definition methods.
* If both the file and parameter methods are specified, then the file method is used.

### Input Files:

* **statmod** (Model):
  Created by the CSMODEL process, this model must include at least one percentile field -
  PCxx \- where xx is the numeric percentile value. It must also include the field MEAN \-
  the average of all the simulated values for each cell.
  Required=Yes

* **cutoff** (Table):
  Allows irregular intervals to be used by specifying cutoff values in this file.
  Required=No

### Output Files:

* **conf_tbl** (Table):
  Output table for displaying confidence for each grade bin defined for successive cutoff
  values.
  Required=Yes

* **conf_plt** (Plot template):
  Output plot template for displaying confidence for each grade bin defined for
  successive cutoff values.
  Required=No

### Fields:

### Parameters:

* **cutint**:
  For regular cutoff grades, this field defines the interval between successive cutoff
  grades. Only required if a CUTOFF file has not been specified.
  Range=0.00001,9999999
  Values=Undefined
  Default=1
  Required=No

* **cutmax**:
  For regular cutoff grades, this field defines the maximum cutoff value. Only required
  if a CUTOFF file has not been specified
  Range=0.00002,9999999
  Values=Undefined
  Default=10
  Required=No

* **plot_tbl**:
  Flag to specify whether a plot data table is output. This contains the data used to
  create the CONF_PLT plot files, and could be used to recreate the plot in other
  software, such as Excel. The plot data table name is the same as the plot file, except
  that "_P" is replaced by "_T".
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **display**:
  Flag to display whether plot files are displayed as the process is run.
  Range=0,1
  Values='0' - do not display plot files. '1' - display plot files as the process is run.
  Default=1
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('cellconf')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.initialize_sandbox(notebook_folder, files_to_copy=files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute cellconf
print("Running cellconf...")
dm_cmd.cellconf(
    statmod_i='t_input_file',  # required
    cutoff_i='optional',  # required
    conf_tbl_o='t_cellconf_out',  # required
    # conf_plt_o='t_cellconf_out',  # optional
    # cutint_p=1,  # optional
    # cutmax_p=10,  # optional
    # plot_tbl_p=0,  # optional
    # display_p=1,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("cellconf execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_cellconf_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")